In [1]:
# conda activate chronocell

import sys

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("code")
# sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/simulations/code")

import Chronocell
from reconstruct_RNA_history import *
# from protein_from_RNA import *

In [2]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.2_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [3]:
Y = traj.X
Q = traj.Q[:, 0, :] 
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo

theta_ = theta.copy()
a0 = theta_[:, 0] # Starting RNA abundance 
a = theta_[:, 1:len(topo.flatten())] 
beta = theta_[:, -2] # Splicing rate
alpha = a * beta[:, None] # These values are divided by splicing rate; removing this factor now
gamma = theta_[:, -1] # Degradation rate
state_grid = np.searchsorted(tau, t, side="left") - 1
state_grid[0] = 0 


In [4]:
# tau
# np.searchsorted(t, tau, side="left")

In [5]:
# U_max = 15
# S_max = 20

# states, index_for = enumerate_states(U_max, S_max)

# # Prep rate matrices

# A_per_gene = [] 
# for j in range(0, alpha.shape[0]):
#     A_for_this_gene = []
#     for i in range(0, alpha.shape[1]):
#         rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
#         A = create_transition_matrix_sparse(rxns, states, index_for, U_max, S_max)
#         A_for_this_gene.append(A)
        
#     A_per_gene.append(A_for_this_gene)

In [6]:
# # Initialize X_fwd with stationary distribution (steady state at t=0)

# pi_per_gene = []

# for j in range(0, alpha.shape[0]):
#     alpha0 = a0[j] * beta[j]
#     rxns0 = define_reactions(alpha0, beta[j], gamma[j])
#     A0 = create_transition_matrix_sparse(rxns0, states, index_for, U_max, S_max)
#     pi = stationary_from_transition_matrix(A0)
#     pi_per_gene.append(pi)

In [7]:
# # Calc forward state probabilities for each gene

# X_fwd_per_gene = []

# for j in range(0, alpha.shape[0]):
#     X_fwd = forward_distribution(A_per_gene[j], pi_per_gene[j], states, t, tau, state_grid)
#     X_fwd_per_gene.append(X_fwd)

In [8]:
# U_max = 300 # np.max(Y[:, :, 0]).astype("int") 
# S_max = 300 # np.max(Y[:, : , 1]).astype("int")
# states, index_for = enumerate_states(U_max, S_max)

In [9]:
X_bw_per_gene = []
states_per_gene = []
    
for j in range(0, Y.shape[0]):
    
    get_expm_per_gene.cache_clear()
    get_A_rev.cache_clear()
    get_expm_rev_per_gene.cache_clear()
    
    print("Starting gene", j)

    # Set max # of RNAs based on observed values for a given gene
    U_max = (np.max(Y[:, j, 0]) + 3).astype("int")
    S_max = (np.max(Y[:, j, 1]) + 3).astype("int")
    states, index_for = enumerate_states(U_max, S_max)
    
    # Make generator matrix (per transcription rate)
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
        A1 = create_transition_matrix(rxns, states, index_for, U_max, S_max)
        A.append(A1)
     
    # Calculate forward probability distribution (needed for reverse generator)
    alpha0 = a0[j] * beta[j]
    rxns0 = define_reactions(alpha0, beta[j], gamma[j])
    A0 = create_transition_matrix(rxns0, states, index_for, U_max, S_max)
    pi = stationary_from_transition_matrix(A0) 
    X_fwd = forward_distribution(A, pi, states, t, tau, state_grid)
    
    global X_fwd_gene
    X_fwd_gene = X_fwd

    # Calculate backward probability distribution (per cell)
    X_bw_per_cell = [] 
    for n in range(0, Q.shape[0]):
        X_bw = backward_distribution(Y, Q, A, X_fwd, j, n, states, index_for, t, tau, state_grid)
        X_bw_per_cell.append(X_bw)
        break
        
    X_bw_per_gene.append(X_bw_per_cell)
    states_per_gene.append(states)

    break

Starting gene 0


In [22]:
cell_idx = 1
gene_idx = 0
j = 0
n = 1

In [10]:
X_bw

array([[3.83954308e-01, 5.19419569e-01, 6.42836272e-01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.34553373e-01, 1.31022867e-01, 1.12243055e-01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [2.35765163e-02, 1.65251683e-02, 9.79915404e-03, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [1.66659021e-29, 2.18961496e-33, 2.99699214e-37, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.84380093e-29, 2.45399500e-33, 3.35758379e-37, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [2.11164142e-29, 2.80124535e-33, 3.81210438e-37, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])

In [24]:
cell_idx

1

In [ ]:
Y[cell_idx, gene_idx, 0], YY[:, gene_idx, 0][cell_idx, gene_idx, 1]

(np.float64(0.0), np.float64(0.0))

In [26]:
np.max(Y[:, gene_idx, 0])

np.float64(23.0)

In [12]:
# Set max # of RNAs based on observed values for a given gene
U_max = (np.max(Y[:, j, 0]) + 3).astype("int")
S_max = (np.max(Y[:, j, 1]) + 3).astype("int")
states, index_for = enumerate_states(U_max, S_max)

# Make generator matrix (per transcription rate)
A = []
for i in range(0, alpha.shape[1]):
    rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
    A1 = create_transition_matrix(rxns, states, index_for, U_max, S_max)
    A.append(A1)

# Calculate forward probability distribution (needed for reverse generator)
alpha0 = a0[j] * beta[j]
rxns0 = define_reactions(alpha0, beta[j], gamma[j])
A0 = create_transition_matrix(rxns0, states, index_for, U_max, S_max)
pi = stationary_from_transition_matrix(A0) # Use stationary distribution at t=0 (system starts in steady state)
X_fwd = forward_distribution(A, pi, states, t, tau, state_grid)

In [16]:
# Initialize backwards trajectory with observed counts
u_curr, s_curr = Y[cell_idx, gene_idx, 0], Y[cell_idx, gene_idx, 1]
x_curr = np.zeros(shape=(A[0].shape[0],), dtype="float")
x_curr[index_for[(u_curr, s_curr)]] = 1.0

# Start backwards trajectory at cell's inferred position in time
t_obs = np.argmax(Q[cell_idx, :])
X_bw = np.zeros(shape=(len(states), len(t))) 
X_bw[:, t_obs] = x_curr

for k in reversed(range(1, t_obs + 1)):
    t_prev, t_curr = t[k-1], t[k]
    state_prev, state_curr = state_grid[k-1], state_grid[k]
    x_curr = X_bw[:, k]
    mu_k = X_fwd[:, k]
        
    if state_prev == state_curr:
        dt = t_curr - t_prev
        A_rev = reverse_generator(A[state_curr], mu_k) 
        x_prev = expm_multiply(A_rev.T * dt, x_curr) 
    
    else:
        # State switch happens in current interval             
        t_s = tau[state_curr]

        # Split backward march into 2 steps
        dt2 = t_curr - t_s # right interval: (state_switch_time, t_k]
        A_rev2 = reverse_generator(A[state_curr], mu_k)
        x_mid = expm_multiply(A_rev2.T * dt2, x_curr) 
        
        dt1 = t_s - t_prev # left interval: (t_{k-1}, state_switch_time]
        A_rev1 = reverse_generator(A[state_prev], mu_k) 
        x_prev = expm_multiply(A_rev1.T * dt1, x_mid) 

    X_bw[:, k-1] = x_prev

In [14]:
expm_multiply(A_rev.T * dt, x_curr) 

array([4.09456956e-132, 2.61648560e-124, 1.13601583e-112, 3.26069575e-099,
       1.32313339e-084, 3.73741666e-069, 5.09038989e-053, 2.79888654e-036,
       6.18600108e-019, 8.31746229e-001, 9.52210503e-123, 2.43398102e-114,
       2.37770598e-102, 1.21327786e-088, 7.69260264e-074, 3.12898548e-058,
       5.80064919e-042, 4.16576451e-025, 1.16533006e-007, 1.15963280e-001,
       4.42880453e-113, 4.52616511e-104, 9.90398564e-092, 8.87885460e-078,
       8.60543176e-063, 4.85394642e-047, 1.14043018e-030, 8.88889027e-014,
       7.12776582e-008, 2.97765460e-002, 3.08974871e-103, 1.26014449e-093,
       6.10885420e-081, 9.38740846e-067, 1.33044279e-051, 9.55714931e-036,
       2.27718816e-019, 1.22331207e-013, 4.53491912e-008, 1.05699625e-002,
       2.87376968e-093, 4.65022750e-083, 4.87288172e-070, 1.22138038e-055,
       2.30055719e-040, 1.64688997e-024, 5.57141319e-019, 1.38367413e-013,
       3.16546991e-008, 4.61662254e-003, 3.33925110e-083, 2.10871764e-072,
       4.53179442e-059, 1

In [ ]:
# Y_observed, Y, theta, rd, true_t, true_l = simulate_RNA(topo, tau, theta[0, :][None, :], n=20000, random_seed=666)